# Notebook 1 - Data Cleaning & Preprocessing
**Owner:** Member 1

**Goal:** Clean the raw dataset and prepare it for EDA and modeling.

**Outputs:**
- `cleaned_insurance.csv` - clean, not encoded (used by EDA)
- `X_train.csv`, `X_test.csv` - unscaled, encoded (used by Decision Tree / Random Forest)
- `X_train_scaled.csv`, `X_test_scaled.csv` - scaled (used by Linear Regression)
- `y_train.csv`, `y_test.csv` - target (original scale)
- `y_train_log.csv`, `y_test_log.csv` - log-transformed target (used by Linear Regression)


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## 2. Load Raw Dataset
All paths assume this notebook runs from inside `notebooks/`.

In [2]:
df = pd.read_csv("../data/raw/insurance.csv")
df.head()


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 3. Initial Inspection

In [3]:
print("Shape:", df.shape)
df.info()
display(df.describe())


Shape: (1338, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 94.5 KB


,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


## 4. Missing Values Check

In [4]:
print("Missing values per column:")
display(df.isnull().sum())


Missing values per column:


age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

## 5. Duplicates Check

In [5]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)


Duplicate rows: 1
Shape after removing duplicates: (1337, 7)


## 6. Outliers Check (Grouped IQR)

`charges` contains two very different populations (smokers vs non-smokers).
Running IQR on the whole column would flag almost all smokers as "outliers" simply
because they belong to a different group, not because their values are wrong.
So we run IQR **separately for each smoker group**.

In [6]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for smoker_value in df["smoker"].unique():
    subset = df.loc[df["smoker"] == smoker_value, "charges"]
    lower, upper = iqr_bounds(subset)
    n_outliers = ((subset < lower) | (subset > upper)).sum()
    print(f"smoker={smoker_value}: {n_outliers} outliers out of {len(subset)} "
          f"(bounds: {lower:.2f} to {upper:.2f})")


smoker=yes: 0 outliers out of 274 (bounds: -9463.20 to 71308.65)
smoker=no: 46 outliers out of 1063 (bounds: -7072.32 to 22424.22)


**Decision:** Even within each smoker group separately, the number of outliers is small
and represents real high-BMI / high-cost patients, not data errors. We keep all rows,
no rows are removed.

## 7. Save Cleaned (Non-Encoded) Copy for EDA

In [7]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/cleaned_insurance.csv", index=False)
print("Saved cleaned_insurance.csv", df.shape)


Saved cleaned_insurance.csv (1337, 7)


## 8. Encoding Categorical Features
- `sex`: male=1, female=0
- `smoker`: yes=1, no=0
- `region`: One-Hot Encoding (no natural order, so LabelEncoder would be wrong here)

In [8]:
df_encoded = df.copy()
df_encoded["sex"] = df_encoded["sex"].map({"male": 1, "female": 0})
df_encoded["smoker"] = df_encoded["smoker"].map({"yes": 1, "no": 0})
df_encoded = pd.get_dummies(df_encoded, columns=["region"], drop_first=True)
df_encoded.head()


,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,False,False,True
1,18,1,33.770,1,0,1725.55230,False,True,False
2,28,1,33.000,3,0,4449.46200,False,True,False
3,33,1,22.705,0,0,21984.47061,True,False,False
4,32,1,28.880,0,0,3866.85520,True,False,False


## 9. Feature Engineering: Interaction Term

EDA showed that the effect of `bmi` on `charges` is much stronger for smokers.
We add `bmi_smoker = bmi * smoker` so Linear Regression (which cannot learn
interactions by itself) can also capture this pattern.

In [9]:
df_encoded["bmi_smoker"] = df_encoded["bmi"] * df_encoded["smoker"]
df_encoded[["bmi", "smoker", "bmi_smoker"]].head()


,bmi,smoker,bmi_smoker
0,27.900,1,27.9
1,33.770,0,0.0
2,33.000,0,0.0
3,22.705,0,0.0
4,28.880,0,0.0


## 10. Feature/Target Split

In [10]:
X = df_encoded.drop("charges", axis=1)
y = df_encoded["charges"]
print("X shape:", X.shape, "| y shape:", y.shape)


X shape: (1337, 9) | y shape: (1337,)


## 11. Train/Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)


Train:

 (1069, 9) | Test: (268, 9)


## 12. Target Transformation (Log) - Tested and Rejected

`charges` is right-skewed, so log-transform seemed like a good idea for Linear Regression.
**We tested it and it made results worse**: R2 dropped from 0.81 to 0.59 (even with a
bias-correction "smearing" adjustment, it stayed worse). Back-transforming with `exp()`
systematically under-predicts the high-cost tail, which is exactly the segment we care
about most (smokers). So we do **not** use log-transform; we keep `charges` on its
original scale for all models.

## 13. Feature Scaling
Linear Regression benefits from scaling. Decision Tree / Random Forest do not need it,
so we keep both versions.

In [12]:
scaler = StandardScaler()
numerical_features = ["age", "bmi", "children", "bmi_smoker"]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

X_train_scaled.head()


,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest,bmi_smoker
1113,-1.157680,1,-0.996928,-0.907908,0,False,False,False,-0.487883
967,-1.300619,1,-0.792762,0.766904,0,False,False,False,-0.487883
598,0.914926,0,1.154664,0.766904,0,True,False,False,-0.487883
170,1.701087,1,1.806837,-0.907908,0,False,True,False,-0.487883
275,0.557580,0,-0.651417,0.766904,0,False,False,False,-0.487883


## 14. Save Scaler and Feature Order
The deployment app needs the exact same scaler and the exact same column order
used here, otherwise predictions will be wrong.

In [13]:
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(list(X_train.columns), "../models/feature_columns.pkl")
print("Saved scaler.pkl and feature_columns.pkl")


Saved scaler.pkl and feature_columns.pkl


## 15. Save Processed Data

In [14]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

X_train_scaled.to_csv("../data/processed/X_train_scaled.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test_scaled.csv", index=False)

print("All processed files saved in ../data/processed/")


All processed files saved in ../data/processed/


## Deliverables - Member 1
- [x] cleaned_insurance.csv (for EDA)
- [x] X_train / X_test (unscaled), X_train_scaled / X_test_scaled
- [x] y_train / y_test, y_train_log / y_test_log
- [x] Documented outlier decision (grouped IQR)
- [x] Interaction feature: bmi_smoker (tested, improves Linear Regression)
- [x] Log-transform tested and rejected (documented why, Section 12)
